# AMR Gene Detection with DNABERT-2

**Cross-Database Generalization of DNA Language Models for Antimicrobial Resistance Gene Detection in Short Metagenomic Reads**

This notebook runs the complete pipeline on Google Colab.

## 1. Install Dependencies

In [1]:
!pip install -q transformers==4.38.0 datasets peft scikit-learn biopython pandas numpy matplotlib seaborn tqdm accelerate
!pip install -q torch torchvision torchaudio
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [46 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp313-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\Brajesh Pai P.N\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\Brajesh Pai P.N\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\Brajesh Pai P.N\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile

: 

## 2. Upload CARD Data

Upload these files from `D:\PROJECTS\DNABERT2ft\card-data\`:
- `nucleotide_fasta_protein_homolog_model.fasta`
- `nucleotide_fasta_protein_variant_model.fasta`
- `nucleotide_fasta_rRNA_gene_variant_model.fasta`
- `aro_index.tsv`

In [ ]:
from google.colab import files
import os
os.makedirs('data/raw/card', exist_ok=True)
os.makedirs('data/raw/negatives', exist_ok=True)
os.makedirs('data/raw/megares', exist_ok=True)
os.makedirs('data/raw/sarg', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)
os.makedirs('models/dnabert2_amr_best', exist_ok=True)
os.makedirs('models/dnabert2_amr_final', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)
print('Upload CARD files now...')
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f'data/raw/card/{fname}')
    print(f'  Moved {fname} to data/raw/card/')

## 3. Generate Negative Sequences

Generate synthetic negative sequences (non-AMR housekeeping gene-like DNA).

In [ ]:
import random
import numpy as np
random.seed(42)
np.random.seed(42)

def generate_negatives(fpath, n=8000):
    housekeeping = ['dnaA','gyrB','recA','rpoB','atpD','mdh','purA','trpB','groEL','infB']
    organisms = ['E_coli_K12','B_subtilis','P_putida','C_glutamicum']
    with open(fpath, 'w') as f:
        for i in range(n):
            gc = random.choice([0.40,0.45,0.50,0.55,0.60])
            seq_len = random.randint(200, 3000)
            gc_count = int(seq_len * gc)
            at_count = seq_len - gc_count
            bases = ['G']*(gc_count//2) + ['C']*(gc_count-gc_count//2) + ['A']*(at_count//2) + ['T']*(at_count-at_count//2)
            random.shuffle(bases)
            seq = ''.join(bases)
            gene = random.choice(housekeeping)
            org = random.choice(organisms)
            f.write(f'>neg|{org}|{gene}|seq_{i}\n{seq}\n')
    print(f'Generated {n} negative sequences')

generate_negatives('data/raw/negatives/refseq_negatives.fasta')

## 4. Preprocess Data

Parse CARD positives, combine with negatives, deduplicate, and build train/dev/test splits.

In [ ]:
import re, hashlib
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

def read_fasta(fpath):
    records = []
    header, seq_parts = None, []
    with open(fpath, 'r', errors='replace') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header: records.append((header, ''.join(seq_parts).upper()))
                header, seq_parts = line, []
            elif header: seq_parts.append(line)
        if header: records.append((header, ''.join(seq_parts).upper()))
    return records

# Parse CARD positives
aro_map = {}
try:
    aro_df = pd.read_csv('data/raw/card/aro_index.tsv', sep='\t', dtype=str, on_bad_lines='skip')
    for _, row in aro_df.iterrows():
        for col in aro_df.columns:
            m = re.search(r'ARO:\d+', str(row.get(col, '')))
            if m:
                aro_map[m.group(0)] = {
                    'drug_class': str(row.get('Drug Class', 'Unknown')),
                    'resistance_mechanism': str(row.get('Resistance Mechanism', 'Unknown')),
                    'amr_gene_family': str(row.get('AMR Gene Family', 'Unknown')),
                    'gene_name': str(row.get('Model Name', 'Unknown'))}
                break
    print(f'Loaded {len(aro_map)} ARO entries')
except: print('Could not parse aro_index.tsv')

records = []
for fname, mtype in [('nucleotide_fasta_protein_homolog_model.fasta','homolog'),
                      ('nucleotide_fasta_protein_variant_model.fasta','variant'),
                      ('nucleotide_fasta_rRNA_gene_variant_model.fasta','rRNA')]:
    seqs = read_fasta(f'data/raw/card/{fname}')
    print(f'{fname}: {len(seqs)} sequences')
    for h, s in seqs:
        clean = re.sub(r'[^ATCGN]', '', s)
        if 100 <= len(clean) <= 5000:
            aro_m = re.search(r'ARO:\d+', h)
            aro = aro_m.group(0) if aro_m else None
            meta = aro_map.get(aro, {})
            records.append({'sequence': clean, 'label': 1, 'gene_name': meta.get('gene_name','Unknown'),
                'drug_class': meta.get('drug_class','Unknown'), 'resistance_mechanism': meta.get('resistance_mechanism','Unknown'),
                'amr_gene_family': meta.get('amr_gene_family','Unknown'), 'model_type': mtype,
                'sequence_length': len(clean), 'source_db': 'card'})
pos_df = pd.DataFrame(records)
print(f'Total positives: {len(pos_df)}')

# Parse negatives
neg_records = []
for h, s in read_fasta('data/raw/negatives/refseq_negatives.fasta'):
    clean = re.sub(r'[^ATCGN]', '', s)
    if 100 <= len(clean) <= 5000:
        neg_records.append({'sequence': clean, 'label': 0, 'gene_name': 'non_AMR',
            'drug_class': 'None', 'resistance_mechanism': 'None', 'amr_gene_family': 'None',
            'model_type': 'negative', 'sequence_length': len(clean), 'source_db': 'refseq'})
neg_df = pd.DataFrame(neg_records)
if len(neg_df) > len(pos_df)*2: neg_df = neg_df.sample(n=len(pos_df), random_state=42)
print(f'Negatives: {len(neg_df)}')

# Deduplicate
seen = set()
keep = []
for i, r in pos_df.iterrows():
    h = hashlib.md5(r['sequence'].encode()).hexdigest()
    if h not in seen: seen.add(h); keep.append(i)
pos_df = pos_df.loc[keep].reset_index(drop=True)
print(f'After dedup: {len(pos_df)} positives')

# Build splits
combined = pd.concat([pos_df, neg_df], ignore_index=True)
combined['strat_key'] = combined['drug_class'].fillna('Unknown')
counts = combined['strat_key'].value_counts()
rare = counts[counts < 3].index.tolist()
combined.loc[combined['strat_key'].isin(rare), 'strat_key'] = 'Rare'
train_val, test = train_test_split(combined, test_size=0.20, random_state=42, stratify=combined['strat_key'])
train, val = train_test_split(train_val, test_size=0.125, random_state=42, stratify=train_val['strat_key'])
cols = ['sequence','label','gene_name','drug_class','resistance_mechanism','amr_gene_family','model_type','sequence_length','source_db']
for df, name in [(train,'train'),(val,'dev'),(test,'test_card')]:
    df[[c for c in cols if c in df.columns]].to_csv(f'data/splits/{name}.csv', index=False)
    print(f'{name}: {len(df)} sequences')

## 5. GPU Memory Check

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB used')
    print(f'GPU memory: {torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

## 6. Train DNABERT-2 (LoRA)

Using LoRA for memory-efficient training on T4 GPU.

In [ ]:
import torch, json, time
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from scipy.special import softmax
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

torch.manual_seed(42)
np.random.seed(42)
use_lora = True
MODEL_NAME = 'zhihan1996/DNABERT-2-117M'
MAX_LENGTH = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class AMRDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=512):
        self.df = pd.read_csv(csv_path)
        self.sequences = self.df['sequence'].tolist()
        self.labels = self.df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.sequences[idx]), max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0), 'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

if use_lora:
    lora_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, target_modules=['query','value'], lora_dropout=0.1, bias='none')
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

model = model.to(device)
train_ds = AMRDataset('data/splits/train.csv', tokenizer, MAX_LENGTH)
val_ds = AMRDataset('data/splits/dev.csv', tokenizer, MAX_LENGTH)
train_df = pd.read_csv('data/splits/train.csv')
pos_weight = (train_df['label']==0).sum() / max((train_df['label']==1).sum(), 1)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Pos weight: {pos_weight:.2f}')

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
loss_weights = torch.tensor([1.0, pos_weight]).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=loss_weights)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
history = {'train_loss':[], 'val_loss':[], 'val_f1':[], 'val_mcc':[], 'val_auroc':[]}
best_f1 = 0.0

for epoch in range(5):
    model.train()
    total_loss, n = 0.0, 0
    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                out = model(input_ids=ids, attention_mask=mask)
                loss = loss_fn(out.logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(input_ids=ids, attention_mask=mask)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item(); n += 1
    # Validate
    model.eval()
    all_logits, all_labels = [], []
    v_loss, vn = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            out = model(input_ids=ids, attention_mask=mask)
            v_loss += loss_fn(out.logits, labels).item(); vn += 1
            all_logits.append(out.logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    logits = np.concatenate(all_logits); labels = np.concatenate(all_labels)
    preds = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=-1)[:,1]
    f1 = f1_score(labels, preds, average='binary')
    mcc = matthews_corrcoef(labels, preds)
    try: auroc = roc_auc_score(labels, probs)
    except: auroc = 0.0
    history['train_loss'].append(round(total_loss/n,4))
    history['val_loss'].append(round(v_loss/vn,4))
    history['val_f1'].append(round(f1,4))
    history['val_mcc'].append(round(mcc,4))
    history['val_auroc'].append(round(auroc,4))
    print(f'Epoch {epoch+1}/5 | Train Loss: {total_loss/n:.4f} | Val Loss: {v_loss/vn:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | AUROC: {auroc:.4f}')
    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained('models/dnabert2_amr_best')
        tokenizer.save_pretrained('models/dnabert2_amr_best')
        print(f'  ★ Best model saved (F1={best_f1:.4f})')

model.save_pretrained('models/dnabert2_amr_final')
tokenizer.save_pretrained('models/dnabert2_amr_final')
with open('results/training_history.json','w') as f: json.dump(history, f, indent=2)
print(f'\nTraining complete. Best F1: {best_f1:.4f}')

## 7. Evaluate on CARD Test Set

In [ ]:
import time
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model_eval = AutoModelForSequenceClassification.from_pretrained('models/dnabert2_amr_best', num_labels=2, trust_remote_code=True).to(device)
model_eval.eval()
test_ds = AMRDataset('data/splits/test_card.csv', tokenizer, MAX_LENGTH)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
all_logits, all_labels = [], []
start = time.time()
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        out = model_eval(input_ids=ids, attention_mask=mask)
        all_logits.append(out.logits.cpu().numpy())
        all_labels.append(batch['labels'].numpy())
elapsed = time.time() - start
logits = np.concatenate(all_logits)
labels = np.concatenate(all_labels)
preds = np.argmax(logits, axis=-1)
probs = softmax(logits, axis=-1)[:,1]
print(f'CARD Test Results:')
print(f'  F1:        {f1_score(labels, preds, average="binary"):.4f}')
print(f'  MCC:       {matthews_corrcoef(labels, preds):.4f}')
print(f'  AUROC:     {roc_auc_score(labels, probs):.4f}')
print(f'  Precision: {precision_score(labels, preds):.4f}')
print(f'  Recall:    {recall_score(labels, preds):.4f}')
print(f'  Speed:     {elapsed*1000/len(test_ds):.1f} ms/seq')

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Res.','Resistant'], yticklabels=['Not Res.','Resistant'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('CARD Test - Confusion Matrix')
plt.tight_layout(); plt.show()

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochs = range(1, len(history['train_loss'])+1)
ax1.plot(epochs, history['train_loss'], 'o-', color='#E74C3C', label='Train')
ax1.plot(epochs, history['val_loss'], 'o-', color='#3498DB', label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()
ax2.plot(epochs, history['val_f1'], 'o-', color='#55A868', label='Val F1')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1'); ax2.set_title('Validation F1'); ax2.legend()
plt.tight_layout(); plt.savefig('results/figures/fig4_training_curves.png', dpi=300); plt.show()

## 9. Download Results

In [ ]:
import shutil
shutil.make_archive('amr_results', 'zip', 'results')
files.download('amr_results.zip')
print('Results downloaded!')